# 08_3_5 DQA Scene Phase2 Temporal Memory

05で効いていた保守的なpseudoGT設定を土台にして、phase2だけを回します。違いは、teacherの一回のconfidenceだけではbbox/cls学習に進ませず、同じtarget画像で同じclassの近いbboxが複数roundで再出現したときだけstable pseudoGTとして昇格する点です。

これはbest checkpointをanchor teacherとして固定するのではなく、各clientのFedSTO local EMA teacherとpseudo-label memoryだけで再現できる設計です。


In [ ]:
from __future__ import annotations

import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_temporal_memory.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
PREFERRED_SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa05_scene_class_profile_5h"
FALLBACK_SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
SOURCE_PHASE1_ROUND = 8
preferred_seed = PREFERRED_SOURCE_WORK_ROOT / "global_checkpoints" / f"phase1_round{SOURCE_PHASE1_ROUND:03d}_global.pt"
fallback_seed = FALLBACK_SOURCE_WORK_ROOT / "global_checkpoints" / f"phase1_round{SOURCE_PHASE1_ROUND:03d}_global.pt"
SOURCE_WORK_ROOT = PREFERRED_SOURCE_WORK_ROOT if preferred_seed.exists() else FALLBACK_SOURCE_WORK_ROOT
WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_5_phase2_temporal_memory"
STATS_ROOT = DQA_ROOT / "stats_dqa08_3_5_phase2_temporal_memory"
LOG_ROOT = DQA_ROOT / "logs_dqa08_3_5_phase2_temporal_memory"
PID_FILE = LOG_ROOT / "train.pid"
RUNNER_LOG = LOG_ROOT / "runner.log"
TRAIN_LOG = LOG_ROOT / "train.log"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT]:
    print(path, "exists=", path.exists())

WORK_ROOT.mkdir(parents=True, exist_ok=True)
STATS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)
print("source seed:", SOURCE_WORK_ROOT / "global_checkpoints" / f"phase1_round{SOURCE_PHASE1_ROUND:03d}_global.pt")
print("workspace:", WORK_ROOT)
print("stats:", STATS_ROOT)
print("logs:", LOG_ROOT)


## Design

- seed: scheduled phase1 round008をround000_warmupとして使う。05のcheckpointが残っていれば05を使い、消えている場合は08のscheduled phase1 round008にfallbackする。どちらもvalidation best選択ではなく固定スケジュールの初期化。
- new pseudoGT: scoreを0.70以下にcapし、high threshold 0.75を超えさせない。objectnessだけ弱く学習。
- matched pseudoGT: 前round memoryとIoU 0.55以上で一致しても、まだscore 0.74以下にcap。
- stable pseudoGT: 2回以上連続して一致したものだけscore 0.78以上に昇格し、bbox + obj + clsの通常pseudoGTとして学習。


In [ ]:
PHASE2_ROUNDS = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT = 29950
MIN_FREE_GIB = 30
MEMORY_IOU = 0.55
MEMORY_MERGE_IOU = 0.70
STABLE_ROUNDS = 2
NEW_SCORE_CAP = 0.70
MATCHED_SCORE_CAP = 0.74
STABLE_SCORE_FLOOR = 0.78
STABLE_OBJ_FLOOR = 0.85
STABLE_CLS_FLOOR = 0.85
MAX_ENTRIES_PER_IMAGE = 80

pd.DataFrame([
    {
        "experiment": "08_3_5_temporal_memory_05profile_r008",
        "phase2_rounds": PHASE2_ROUNDS,
        "batch_size": BATCH_SIZE,
        "workers": WORKERS,
        "seed": f"{SOURCE_WORK_ROOT.name}_phase1_round{SOURCE_PHASE1_ROUND:03d}",
        "low/high": "0.35 / 0.75",
        "new/matched/stable": f"{NEW_SCORE_CAP} / {MATCHED_SCORE_CAP} / {STABLE_SCORE_FLOOR}",
        "memory_iou": MEMORY_IOU,
        "stable_rounds": STABLE_ROUNDS,
    }
])


In [ ]:
def pid_state(pid: int | None) -> str:
    if pid is None:
        return "not started"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "not running"
    except PermissionError:
        return "running (no permission)"
    stat_path = Path(f"/proc/{pid}/stat")
    if stat_path.exists():
        try:
            state = stat_path.read_text().split()[2]
            if state == "Z":
                return "not running (zombie)"
        except Exception:
            pass
    return "running"

def read_pid() -> int | None:
    if not PID_FILE.exists():
        return None
    try:
        return int(PID_FILE.read_text().strip())
    except ValueError:
        return None

def tail_lines(path: Path, n: int = 80) -> str:
    if not path.exists():
        return f"{path} does not exist"
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])

def history_rows() -> list[dict]:
    path = WORK_ROOT / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text(encoding="utf-8"))

def latest_checkpoint() -> Path:
    rows = history_rows()
    if rows:
        return Path(rows[-1]["global"])
    return WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"

def train_env() -> dict[str, str]:
    env = os.environ.copy()
    env.update({
        "DQA0835_SOURCE_WORK_ROOT": str(SOURCE_WORK_ROOT),
        "DQA0835_SOURCE_PHASE1_ROUND": str(SOURCE_PHASE1_ROUND),
        "DQA0835_MEMORY_ROOT": str(STATS_ROOT / "pseudo_memory"),
        "DQA0835_MEMORY_IOU": str(MEMORY_IOU),
        "DQA0835_MEMORY_MERGE_IOU": str(MEMORY_MERGE_IOU),
        "DQA0835_STABLE_ROUNDS": str(STABLE_ROUNDS),
        "DQA0835_NEW_SCORE_CAP": str(NEW_SCORE_CAP),
        "DQA0835_MATCHED_SCORE_CAP": str(MATCHED_SCORE_CAP),
        "DQA0835_STABLE_SCORE_FLOOR": str(STABLE_SCORE_FLOOR),
        "DQA0835_STABLE_OBJ_FLOOR": str(STABLE_OBJ_FLOOR),
        "DQA0835_STABLE_CLS_FLOOR": str(STABLE_CLS_FLOOR),
        "DQA0835_MAX_ENTRIES_PER_IMAGE": str(MAX_ENTRIES_PER_IMAGE),
        "DQA0835_STATS_QUALITY_MODE": "confidence",
        "ET_SKIP_AFTER_TRAIN_BEST_VAL": "1",
    })
    return env

def train_cmd() -> list[str]:
    return [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root", str(WORK_ROOT),
        "--stats-root", str(STATS_ROOT),
        "--phase1-rounds", "0",
        "--phase2-rounds", str(PHASE2_ROUNDS),
        "--warmup-epochs", "0",
        "--batch-size", str(BATCH_SIZE),
        "--workers", str(WORKERS),
        "--gpus", str(GPUS),
        "--master-port", str(MASTER_PORT),
        "--dqa-start-phase", "2",
        "--min-free-gib", str(MIN_FREE_GIB),
        "--log-file", str(TRAIN_LOG),
        "--append-train-log",
    ]

print(" ".join(train_cmd()))


## Start Training

このセルはバックグラウンドで08_3_5を開始します。既にPIDが生きている場合は二重起動しません。


In [ ]:
pid = read_pid()
state = pid_state(pid)
print("current pid:", pid, state)

if state == "running":
    print("already running")
else:
    LOG_ROOT.mkdir(parents=True, exist_ok=True)
    with RUNNER_LOG.open("ab") as log:
        proc = subprocess.Popen(
            train_cmd(),
            cwd=REPO_ROOT,
            env=train_env(),
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    PID_FILE.write_text(str(proc.pid), encoding="utf-8")
    print("started pid:", proc.pid)
    print("runner log:", RUNNER_LOG)
    print("train log:", TRAIN_LOG)


## Progress

学習中はこのセルを何度か実行すると、完了round、最新checkpoint、pseudo memoryの増え方、ログ末尾を確認できます。


In [ ]:
pid = read_pid()
print("pid:", pid, pid_state(pid))
rows = history_rows()
print("completed rounds:", len(rows), "/", PHASE2_ROUNDS)
print("latest checkpoint:", latest_checkpoint())

if rows:
    display(pd.DataFrame(rows).tail(12))

memory_rows = []
for path in sorted((STATS_ROOT / "pseudo_memory").glob("pseudo_memory_client*.json")):
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        boxes = sum(len(v) for v in payload.values() if isinstance(v, list))
        memory_rows.append({"file": path.name, "images": len(payload), "boxes": boxes, "mb": round(path.stat().st_size / 1024 / 1024, 2)})
    except Exception as exc:
        memory_rows.append({"file": path.name, "error": str(exc)})
if memory_rows:
    display(pd.DataFrame(memory_rows))

stat_rows = []
for path in sorted(STATS_ROOT.glob("phase2_round*.json")):
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        clients = payload.get("clients", []) if isinstance(payload, dict) else []
        total = 0.0
        qsum = 0.0
        for client in clients:
            counts = [float(x) for x in client.get("counts", [])]
            quality = [float(x) for x in client.get("mean_quality_scores", [])]
            total += sum(counts)
            qsum += sum(c * q for c, q in zip(counts, quality))
        stat_rows.append({"round": path.stem, "pseudo_total": int(total), "mean_quality": round(qsum / total, 4) if total else 0.0})
    except Exception as exc:
        stat_rows.append({"round": path.stem, "error": str(exc)})
if stat_rows:
    display(pd.DataFrame(stat_rows).tail(12))

print("\n--- runner log tail ---")
print(tail_lines(RUNNER_LOG, 80))
print("\n--- train log tail ---")
print(tail_lines(TRAIN_LOG, 80))


## Evaluation

学習が進んだら、source checkpoint、残っている05 checkpoint、08_3_5の各roundをpaper protocolのtotal splitで比較します。


In [ ]:
candidate_checkpoints = [
    ("source_warmup", SOURCE_WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"),
    ("source_phase1_r008", SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round008_global.pt"),
    ("source_phase2_r002", SOURCE_WORK_ROOT / "global_checkpoints" / "phase2_round002_global.pt"),
    ("dqa05_warmup", PREFERRED_SOURCE_WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"),
    ("dqa05_phase1_r008", PREFERRED_SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round008_global.pt"),
    ("dqa05_phase2_r002", PREFERRED_SOURCE_WORK_ROOT / "global_checkpoints" / "phase2_round002_global.pt"),
]
checkpoints = []
seen_paths = set()
for label, path in candidate_checkpoints:
    if path.exists() and path not in seen_paths:
        checkpoints.append((label, path))
        seen_paths.add(path)
for r in [1, 2, 3, 5, 8, 10]:
    ckpt = WORK_ROOT / "global_checkpoints" / f"phase2_round{r:03d}_global.pt"
    if ckpt.exists():
        checkpoints.append((f"dqa0835_phase2_r{r:03d}", ckpt))

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace", str(WORK_ROOT),
    "--splits", "total",
    "--batch-size", "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError(f"evaluation failed: {result.returncode}")

summary_csv = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
df = pd.read_csv(summary_csv)
cols = ["checkpoint_label", "split", "precision", "recall", "map50", "map50_95", "status"]
display(df[cols].sort_values("map50_95", ascending=False))


In [ ]:
# 必要なときだけ使う停止セルです。
# pid = read_pid()
# if pid and pid_state(pid) == "running":
#     os.killpg(pid, signal.SIGTERM)
#     print("sent SIGTERM to", pid)
